# Решения: функция-предсказатель

**Для преподавателя.** Все задачи из `lesson.ipynb` и `homework.ipynb`.

In [ ]:
AREAS_SQM = [28, 32, 45, 55, 60]
PRICES_MLN = [3.9, 4.2, 5.8, 6.5, 7.1]
ROOMS = [1, 1, 2, 2, 3]


## Урок. 1. `predict_price`

In [ ]:
def predict_price(area_sqm, intercept=1.5, coef=0.09):
    return intercept + coef * area_sqm


assert predict_price(0) == 1.5
assert abs(predict_price(50) - 6.0) < 1e-9

## Урок. 2. print vs return

In [ ]:
def predict_print(area_sqm):
    print(predict_price(area_sqm))


def predict_return(area_sqm):
    return predict_price(area_sqm)


print('predict_print(45) + 1 →', end=' ')
try:
    print(predict_print(45) + 1)
except TypeError:
    print('ошибка: print не возвращает значение (None)')

print('predict_return(45) + 1 =', predict_return(45) + 1)

## Урок. 3. `batch_predict`

In [ ]:
def batch_predict(values, predict_fn):
    return [predict_fn(v) for v in values]


areas = list(AREAS_SQM)
preds = batch_predict(areas, predict_price)
assert len(preds) == len(areas)
assert batch_predict([], predict_price) == []

## Урок. 4. `mae`

При разной длине списков — `ValueError`: метрика не определена.

In [ ]:
def mae(preds, facts):
    if len(preds) != len(facts):
        raise ValueError('preds и facts должны быть одной длины')
    if not preds:
        return 0.0
    return sum(abs(p - f) for p, f in zip(preds, facts)) / len(preds)


assert mae([2, 5], [3, 3]) == 1.5
print('MAE на паре:', mae(preds, PRICES_MLN))

## Урок. 5. Выбор коэффициента

`coef=coef` в аргументах по умолчанию — иначе замыкание в цикле даст один и тот же coef.

In [ ]:
results = []

for coef_percent in range(5, 16):
    coef = coef_percent / 100

    def candidate(area_sqm, coef=coef):
        return predict_price(area_sqm, intercept=1.5, coef=coef)

    predictions = batch_predict(AREAS_SQM, candidate)
    error = mae(predictions, PRICES_MLN)
    results.append((error, coef, predictions))

best_error, best_coef, best_predictions = min(results)
print(f'Лучший коэффициент: {best_coef:.2f}; MAE: {best_error:.3f}')

## ДЗ. 1. Базовые функции — см. ячейки урока выше

## ДЗ. 2. `predict_room_price`

In [ ]:
def predict_room_price(area_sqm, rooms, intercept=1.5, coef=0.09):
    room_bonus = max(rooms - 1, 0) * 0.3
    return predict_price(area_sqm, intercept, coef) + room_bonus


assert abs(predict_room_price(45, 2) - 5.85) < 1e-9
print('строка 2:', predict_room_price(AREAS_SQM[2], ROOMS[2]))

## ДЗ. 3. Устойчивость при удалении точки

In [ ]:
idx_max = max(range(len(AREAS_SQM)), key=lambda i: AREAS_SQM[i])
areas_sub = [a for i, a in enumerate(AREAS_SQM) if i != idx_max]
prices_sub = [p for i, p in enumerate(PRICES_MLN) if i != idx_max]

sub_results = []
for coef_percent in range(5, 16):
    coef = coef_percent / 100

    def candidate(area_sqm, coef=coef):
        return predict_price(area_sqm, intercept=1.5, coef=coef)

    predictions = batch_predict(areas_sub, candidate)
    sub_results.append((mae(predictions, prices_sub), coef))

best_sub_error, best_sub_coef = min(sub_results)
print(f'Убрали {AREAS_SQM[idx_max]} м²: coef={best_sub_coef:.2f}, MAE={best_sub_error:.3f}')
print(
    'На этих данных лучший coef совпал (0.09), но MAE чуть ниже — '
    'одна точка влияет на метрику.'
)

## ДЗ. 4. Сетка intercept × coef

In [ ]:
grid = []
for intercept in [0.5, 1.0, 1.5, 2.0]:
    for coef_percent in range(5, 16):
        coef = coef_percent / 100

        def candidate(area_sqm, intercept=intercept, coef=coef):
            return predict_price(area_sqm, intercept, coef)

        preds = batch_predict(AREAS_SQM, candidate)
        grid.append((mae(preds, PRICES_MLN), intercept, coef))

top3 = sorted(grid)[:3]
for error, intercept, coef in top3:
    print(f'MAE={error:.3f}, intercept={intercept}, coef={coef:.2f}')